# Download XL-Sum News Articles

Pull BBC news articles from XL-Sum in English, French, Turkish, Chinese.
Run this on Colab (datasets library works there), save to Drive.

In [ ]:
!pip install -q datasets

import json, os
from pathlib import Path
from datasets import load_dataset

from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = Path('/content/drive/MyDrive/LRTIA/Data/xlsum_news')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LANGUAGES = {
    'en': 'english',
    'fr': 'french',
    'tr': 'turkish',
    'zh': 'chinese_simplified',
}

MIN_CHARS = 2000  # news articles are shorter than wiki
MAX_ARTICLES = 60

for lang_code, xlsum_name in LANGUAGES.items():
    print(f'\n--- {xlsum_name} ---')
    ds = load_dataset('csebuetnlp/xlsum', xlsum_name, split='train', trust_remote_code=True)
    print(f'  Total: {len(ds)}')

    docs = []
    for article in ds:
        text = article['text'].strip()
        if len(text) < MIN_CHARS:
            continue
        docs.append({
            'doc_id': f'News_{lang_code}_{len(docs):04d}',
            'author_id': f'xlsum_{lang_code}',
            'domain': 'news',
            'population': f'written_{lang_code}',
            'text': text,
            'metadata': json.dumps({
                'dataset': 'XLSum',
                'language': lang_code,
                'title': article.get('title', ''),
                'char_count': len(text),
            }, ensure_ascii=False),
        })
        if len(docs) >= MAX_ARTICLES:
            break

    out_path = OUTPUT_DIR / f'{lang_code}_news.jsonl'
    with open(out_path, 'w') as f:
        for doc in docs:
            f.write(json.dumps(doc, ensure_ascii=False) + '\n')

    chars = [len(d['text']) for d in docs]
    print(f'  Kept: {len(docs)} articles (>={MIN_CHARS} chars)')
    print(f'  Chars: min={min(chars)}, max={max(chars)}, mean={sum(chars)//len(chars)}')

print('\nDone!')